CRSP Mutual Fund Data Preparation

This notebook prepares the raw CRSP mutual fund data for the empirical analysis of my bachelor thesis on mutual fund performance and fund flows.

The goal of this first step is to:
1. Load the raw CRSP monthly returns, fund summary, and fund-portfolio map files.
2. Inspect the structure and columns of each dataset.
3. Create column overview tables.
4. Prepare the basis for constructing a portfolio-month dataset.

In [1]:
# ============================================================
# 0. Imports & project setup
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ── Project parameters ───────────────────────────────────────
PROJECT_NAME   = "CRSP Analysis"
BASE_DIR       = Path(".")                              # notebook location
DATA_DIR       = BASE_DIR / "real, clear data" / "raw" # raw CSV files
PROCESSED_DIR  = BASE_DIR / "processed"
OUTPUT_DIR     = BASE_DIR / "output"

# Create output folders if they don't exist
PROCESSED_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project : {PROJECT_NAME}")
print(f"Base dir: {BASE_DIR.resolve()}")
print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Processed → {PROCESSED_DIR.resolve()}")
print(f"Output    → {OUTPUT_DIR.resolve()}")

# ── Explicitly name the three CRSP raw files ─────────────────
FILE_RETURNS  = DATA_DIR / "monthly_returns_2000_2025.csv"
FILE_SUMMARY  = DATA_DIR / "fund_summary_quarterly_2000_2025.csv"
FILE_PORTMAP  = DATA_DIR / "fund_portfolio_map_2000_2025.csv"

for fp in [FILE_RETURNS, FILE_SUMMARY, FILE_PORTMAP]:
    if not fp.exists():
        raise FileNotFoundError(f"Raw file not found: {fp.resolve()}")

print(f"\nRaw files located:")
print(f"  Returns  : {FILE_RETURNS.name}")
print(f"  Summary  : {FILE_SUMMARY.name}")
print(f"  Port map : {FILE_PORTMAP.name}")

# ============================================================
# 1. Load the three CSV files
# ============================================================
# monthly_returns  — caldt, crsp_fundno, mtna, mret, mnav
monthly_returns = pd.read_csv(FILE_RETURNS, low_memory=False,
                               parse_dates=['caldt'])

# fund_summary     — quarterly snapshot: exp_ratio, turn_ratio,
#                    mgmt_name, lipper_obj_name, crsp_obj_cd, …
fund_summary    = pd.read_csv(FILE_SUMMARY, low_memory=False,
                               parse_dates=['caldt', 'first_offer_dt'])

# fund_portfolio_map — maps crsp_fundno → crsp_portno (share-class → portfolio)
fund_portmap    = pd.read_csv(FILE_PORTMAP, low_memory=False,
                               parse_dates=['begdt', 'enddt'])

# ── Shapes ───────────────────────────────────────────────────
print("\n── Shapes ──────────────────────────────────────────────")
print(f"  {FILE_RETURNS.name:45s}  {str(monthly_returns.shape):>15}")
print(f"  {FILE_SUMMARY.name:45s}  {str(fund_summary.shape):>15}")
print(f"  {FILE_PORTMAP.name:45s}  {str(fund_portmap.shape):>15}")

# ── First rows ───────────────────────────────────────────────
print(f"\n── Head: {FILE_RETURNS.name} ──")
display(monthly_returns.head())

print(f"\n── Head: {FILE_SUMMARY.name} ──")
display(fund_summary.head())

print(f"\n── Head: {FILE_PORTMAP.name} ──")
display(fund_portmap.head())

# ============================================================
# 2. Column overview tables
# ============================================================
def column_overview(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Return a tidy summary table for every column in *df*."""
    overview = pd.DataFrame({
        "column"    : df.columns,
        "dtype"     : df.dtypes.values,
        "non_null"  : df.notna().sum().values,
        "null"      : df.isna().sum().values,
        "null_pct"  : (df.isna().mean() * 100).round(2).values,
        "n_unique"  : df.nunique().values,
        "sample_val": [df[c].dropna().iloc[0] if df[c].notna().any() else np.nan
                       for c in df.columns],
    })
    overview.insert(0, "file", label)
    return overview.reset_index(drop=True)

overview_returns = column_overview(monthly_returns, FILE_RETURNS.name)
overview_summary = column_overview(fund_summary,    FILE_SUMMARY.name)
overview_portmap = column_overview(fund_portmap,    FILE_PORTMAP.name)

# Display
print(f"\n── Column overview: {FILE_RETURNS.name} ──")
display(overview_returns)

print(f"\n── Column overview: {FILE_SUMMARY.name} ──")
display(overview_summary)

print(f"\n── Column overview: {FILE_PORTMAP.name} ──")
display(overview_portmap)

# Save to processed folder
overview_returns.to_csv(PROCESSED_DIR / f"col_overview_{FILE_RETURNS.stem}.csv", index=False)
overview_summary.to_csv(PROCESSED_DIR / f"col_overview_{FILE_SUMMARY.stem}.csv", index=False)
overview_portmap.to_csv(PROCESSED_DIR / f"col_overview_{FILE_PORTMAP.stem}.csv", index=False)

print("\nColumn overview tables saved to:", PROCESSED_DIR.resolve())


Project : CRSP Analysis
Base dir: /Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data
Data dir: /Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data/real, clear data/raw
Processed → /Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data/processed
Output    → /Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data/output

Raw files located:
  Returns  : monthly_returns_2000_2025.csv
  Summary  : fund_summary_quarterly_2000_2025.csv
  Port map : fund_portfolio_map_2000_2025.csv

── Shapes ──────────────────────────────────────────────
  monthly_returns_2000_2025.csv                     (8690724, 5)
  fund_summary_quarterly_2000_2025.csv             (2801929, 26)
  fund_portfolio_map_2000_2025.csv                    (81068, 7)

── Head: monthly_returns_2000_2025.csv ──


,caldt,crsp_fundno,mtna,mret,mnav
0,2000-01-31,1,190.800,-0.004564,13.82
1,2000-02-29,1,186.100,0.004245,13.80
2,2000-03-31,1,185.900,0.012627,13.89
3,2000-04-28,1,181.900,-0.005027,13.74
4,2000-05-31,1,179.700,-0.003660,13.61



── Head: fund_summary_quarterly_2000_2025.csv ──


,summary_period2,crsp_fundno,caldt,fund_name,ticker,mgmt_name,index_fund_flag,et_flag,delist_cd,first_offer_dt,...,fiscal_yearend,crsp_obj_cd,si_obj_cd,wbrger_obj_cd,policy,lipper_class,lipper_class_name,lipper_obj_cd,lipper_obj_name,lipper_asset_cd
0,Q,1,2000-03-31,AARP Income Trust: AARP Bond Fund for Income,AABIX,AMERICAN ASSOC OF RETIRED PERSONS,NaN,NaN,M,1997-02-01,...,NaN,ICQY,NaN,NaN,NaN,BBB,Corporate Debt Funds BBB-Rated,BBB,Corporate Debt Funds BBB-Rated,TX
1,Q,1,2000-06-30,AARP Income Trust: AARP Bond Fund for Income,AABIX,AMERICAN ASSOC OF RETIRED PERSONS,NaN,NaN,M,1997-02-01,...,NaN,ICQY,NaN,NaN,NaN,BBB,Corporate Debt Funds BBB-Rated,BBB,Corporate Debt Funds BBB-Rated,TX
2,Q,1,2000-09-29,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q,2,2000-03-31,AARP Managed Investment Portfolios Trust: AARP...,AADGX,AMERICAN ASSOC OF RETIRED PERSONS,NaN,NaN,M,1997-02-03,...,NaN,EDYB,NaN,NaN,NaN,MLVE,Multi-Cap Value Funds,GI,Growth and Income Funds,EQ
4,Q,2,2000-06-30,AARP Managed Investment Portfolios Trust: AARP...,AADGX,AMERICAN ASSOC OF RETIRED PERSONS,NaN,NaN,M,1997-02-03,...,NaN,EDYB,NaN,NaN,NaN,MLVE,Multi-Cap Value Funds,GI,Growth and Income Funds,EQ



── Head: fund_portfolio_map_2000_2025.csv ──


...


── Column overview: fund_summary_quarterly_2000_2025.csv ──


,file,column,dtype,non_null,null,null_pct,n_unique,sample_val
0,fund_summary_quarterly_2000_2025.csv,summary_period2,str,2801929,0,0.00,2,Q
1,fund_summary_quarterly_2000_2025.csv,crsp_fundno,int64,2801929,0,0.00,70559,1
2,fund_summary_quarterly_2000_2025.csv,caldt,datetime64[us],2801929,0,0.00,104,2000-03-31 00:00:00
3,fund_summary_quarterly_2000_2025.csv,fund_name,str,2781138,20791,0.74,153455,AARP Income Trust: AARP Bond Fund for Income
4,fund_summary_quarterly_2000_2025.csv,ticker,str,2442376,359553,12.83,60989,AABIX
5,fund_summary_quarterly_2000_2025.csv,mgmt_name,str,2756205,45724,1.63,6278,AMERICAN ASSOC OF RETIRED PERSONS
6,fund_summary_quarterly_2000_2025.csv,index_fund_flag,str,263859,2538070,90.58,3,B
7,fund_summary_quarterly_2000_2025.csv,et_flag,str,165632,2636297,94.09,2,F
8,fund_summary_quarterly_2000_2025.csv,delist_cd,str,137976,2663953,95.08,2,M
9,fund_summary_quarterly_2000_2025.csv,first_offer_dt,datetime64[us],2780604,21325,0.76,8727,1997-02-01 00:00:00



── Column overview: fund_portfolio_map_2000_2025.csv ──


,file,column,dtype,non_null,null,null_pct,n_unique,sample_val
0,fund_portfolio_map_2000_2025.csv,crsp_fundno,int64,81068,0,0.00,61870,4273
1,fund_portfolio_map_2000_2025.csv,crsp_portno,int64,81068,0,0.00,30362,1000001
2,fund_portfolio_map_2000_2025.csv,begdt,datetime64[us],81068,0,0.00,273,2003-03-31 00:00:00
3,fund_portfolio_map_2000_2025.csv,enddt,datetime64[us],81068,0,0.00,124,2026-03-31 00:00:00
4,fund_portfolio_map_2000_2025.csv,crsp_cl_grp,float64,65536,15532,19.16,12519,2000866.0
5,fund_portfolio_map_2000_2025.csv,fund_name,str,81068,0,0.00,61687,"AB Cap Fund, Inc: AB Small Cap Growth Portfoli..."
6,fund_portfolio_map_2000_2025.csv,ticker,str,66374,14694,18.13,47801,QUASX



Column overview tables saved to: /Users/justin.hahn/Downloads/Uni /Bachlorarbeit /code/Bachelorarbeit/real data/processed
